# Part A (Optimised): CNNs on CIFAR-100

> **Goal:** Push accuracy well above the 42% baseline through five targeted upgrades:
> 1. Per-channel normalisation
> 2. Advanced augmentation: RandAugment + CutMix + Mixup
> 3. Deeper ResNet-style architecture with BatchNorm + SE attention
> 4. Cosine-annealing LR schedule with warm restarts + label smoothing
> 5. Fine-tuned EfficientNetV2-S (two-phase transfer learning)


## 1. Environment Setup

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar100
from tensorflow.keras import layers, models, regularizers, callbacks
from tensorflow.keras.applications import EfficientNetV2S
import numpy as np
import matplotlib.pyplot as plt
import math

print(f'TensorFlow version: {tf.__version__}')
print(f'GPUs available: {tf.config.list_physical_devices("GPU")}')

tf.random.set_seed(42)
np.random.seed(42)


## 2. Data Loading & Normalisation

**Improvement over baseline:** Per-channel mean/std normalisation (ImageNet-style) instead of simple `/255`.
This centres activations and accelerates convergence by ~1-2 pp.


In [ ]:
(x_train, y_train_raw), (x_test, y_test_raw) = cifar100.load_data()

x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

# Per-channel stats from training set only (no data leakage)
mean = x_train.mean(axis=(0, 1, 2))
std  = x_train.std(axis=(0, 1, 2))

x_train = (x_train - mean) / (std + 1e-7)
x_test  = (x_test  - mean) / (std + 1e-7)

# Integer labels (compatible with SparseCategoricalCrossentropy + label smoothing)
y_train = y_train_raw.flatten()
y_test  = y_test_raw.flatten()

print(f'x_train: {x_train.shape}, channel means: {x_train.mean(axis=(0,1,2)).round(4)}')
print(f'x_test : {x_test.shape}')


## 3. Advanced Data Augmentation

**Improvements over baseline:**

| Aug. technique | Baseline (A1) | Optimised |
|---|---|---|
| Random horizontal flip | No | Yes |
| Random crop/translation | No | Yes (10%) |
| Random rotation | No | Yes (±36°) |
| Colour/contrast jitter | No | Yes |
| **CutMix** | No | Yes |
| **Mixup** | No | Yes |

**CutMix:** Replaces a random rectangular patch of one image with the same patch from another.
Forces the model to learn from partial objects and improves calibration.

**Mixup:** Linear interpolation of two images and their labels:
$\tilde{x} = \lambda x_i + (1-\lambda)x_j$,  $\lambda \sim \text{Beta}(\alpha,\alpha)$


In [ ]:
# Standard Keras augmentation layers (GPU-accelerated, fused into model graph)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name='augmentation')

# ── CutMix ────────────────────────────────────────────────────────────────
def cutmix(images, labels, alpha=1.0):
    batch_size = tf.shape(images)[0]
    lam = tf.cast(np.random.beta(alpha, alpha), tf.float32)
    H, W = 32, 32
    cut_rat = tf.sqrt(1.0 - lam)
    cut_w = tf.cast(W * cut_rat, tf.int32)
    cut_h = tf.cast(H * cut_rat, tf.int32)
    cx = tf.random.uniform((), 0, W, tf.int32)
    cy = tf.random.uniform((), 0, H, tf.int32)
    x1 = tf.clip_by_value(cx - cut_w // 2, 0, W)
    y1 = tf.clip_by_value(cy - cut_h // 2, 0, H)
    x2 = tf.clip_by_value(cx + cut_w // 2, 0, W)
    y2 = tf.clip_by_value(cy + cut_h // 2, 0, H)
    indices = tf.random.shuffle(tf.range(batch_size))
    shuffled = tf.gather(images, indices)
    # Build binary mask: 0 in cut region, 1 elsewhere
    mask = tf.ones([H, W, 1], tf.float32)
    row_mask = tf.logical_and(tf.range(H) >= y1, tf.range(H) < y2)
    col_mask = tf.logical_and(tf.range(W) >= x1, tf.range(W) < x2)
    patch_mask = tf.cast(
        tf.logical_not(tf.logical_and(
            tf.reshape(row_mask, [H, 1, 1]),
            tf.reshape(col_mask, [1, W, 1])
        )), tf.float32)
    mixed = images * patch_mask + shuffled * (1.0 - patch_mask)
    lam_adj = 1.0 - tf.cast((x2 - x1) * (y2 - y1), tf.float32) / float(H * W)
    return mixed, (labels, tf.gather(labels, indices), lam_adj)

# ── Mixup ─────────────────────────────────────────────────────────────────
def mixup(images, labels, alpha=0.2):
    batch_size = tf.shape(images)[0]
    lam = tf.cast(np.random.beta(alpha, alpha), tf.float32)
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed = lam * images + (1.0 - lam) * tf.gather(images, indices)
    return mixed, (labels, tf.gather(labels, indices), lam)

# ── Mixed loss (label smoothing + interpolated targets) ───────────────────
def mixed_loss(y_true_tuple, y_pred, num_classes=100, smoothing=0.1):
    labels_a, labels_b, lam = y_true_tuple
    def smooth_xent(y, pred):
        y_oh = tf.one_hot(y, num_classes)
        y_sm = y_oh * (1.0 - smoothing) + smoothing / num_classes
        return -tf.reduce_mean(tf.reduce_sum(y_sm * tf.math.log(pred + 1e-7), axis=-1))
    return lam * smooth_xent(labels_a, y_pred) + (1.0 - lam) * smooth_xent(labels_b, y_pred)

print('Augmentation pipeline ready.')


## 4. Optimised Architecture: ResNet + Squeeze-and-Excitation + BatchNorm

**Key improvements over A1 baseline:**

| Feature | Baseline (A1) | Optimised |
|---|---|---|
| Depth | 3 conv blocks | 4 residual stages (8 blocks) |
| Filters | 32 → 64 → 128 | 64 → 128 → 256 → 512 |
| Normalisation | None | BatchNorm after every conv |
| Skip connections | None | Residual shortcuts |
| Channel attention | None | Squeeze-and-Excitation (SE) |
| Regularisation | Dropout(0.5) | L2 weight decay + Dropout(0.3) |
| Classifier head | Dense(128) | GAP → Dense(256) → Dense(100) |

**Squeeze-and-Excitation block:** Learns per-channel importance weights by 
globally pooling spatial info, passing through a small MLP, and scaling channels:
$\hat{\mathbf{x}}_c = \sigma(W_2 \cdot \text{ReLU}(W_1 \cdot \text{GAP}(\mathbf{x}))) \cdot \mathbf{x}_c$


In [ ]:
def residual_block(x, filters, stride=1, l2_reg=1e-4):
    """Pre-activation ResNet block with SE channel attention."""
    shortcut = x
    # Main path: BN -> ReLU -> Conv -> BN -> ReLU -> Conv
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, strides=stride, padding='same',
                      kernel_regularizer=regularizers.l2(l2_reg), use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same',
                      kernel_regularizer=regularizers.l2(l2_reg), use_bias=False)(x)
    # Squeeze-and-Excitation channel attention
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(max(filters // 16, 4), activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1, 1, filters))(se)
    x  = layers.Multiply()([x, se])
    # Projection shortcut when shape changes
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same',
                                 kernel_regularizer=regularizers.l2(l2_reg),
                                 use_bias=False)(shortcut)
    return layers.Add()([x, shortcut])


def build_resnet_cifar(num_classes=100, l2_reg=1e-4):
    inp = layers.Input(shape=(32, 32, 3))
    # Stem: single conv, no pooling (preserve 32x32)
    x = layers.Conv2D(64, 3, padding='same',
                      kernel_regularizer=regularizers.l2(l2_reg), use_bias=False)(inp)
    # Stage 1: 32x32, 64 filters x2
    x = residual_block(x, 64,  stride=1)
    x = residual_block(x, 64,  stride=1)
    # Stage 2: 16x16, 128 filters x2
    x = residual_block(x, 128, stride=2)
    x = residual_block(x, 128, stride=1)
    # Stage 3: 8x8, 256 filters x2
    x = residual_block(x, 256, stride=2)
    x = residual_block(x, 256, stride=1)
    # Stage 4: 4x4, 512 filters x2
    x = residual_block(x, 512, stride=2)
    x = residual_block(x, 512, stride=1)
    # Classifier head
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inp, out, name='ResNet_SE_CIFAR100')


model_opt = build_resnet_cifar()
model_opt.summary()
print(f'\nTotal params: {model_opt.count_params():,}')


## 5. Learning Rate Schedule: Warmup + Cosine Annealing with Warm Restarts

**Improvement over baseline:** Fixed Adam LR=0.001 for 80 epochs.

We use **SGDR (Stochastic Gradient Descent with Warm Restarts)**:
$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max}-\eta_{\min})\left(1 + \cos\left(\frac{\pi\, T_{cur}}{T_i}\right)\right)$$

Warm restarts help escape local minima by periodically resetting the LR.
A linear warmup phase stabilises early training when weights are random.


In [ ]:
EPOCHS        = 150
WARMUP_EPOCHS = 5
BASE_LR       = 1e-3
MIN_LR        = 1e-6
T_RESTART     = 50   # cosine period in epochs
BATCH_SIZE    = 128
NUM_CLASSES   = 100
SMOOTHING     = 0.1

# Visualise the schedule before training
lrs = []
for ep in range(EPOCHS):
    if ep < WARMUP_EPOCHS:
        lrs.append(BASE_LR * (ep + 1) / WARMUP_EPOCHS)
    else:
        t = (ep - WARMUP_EPOCHS) % T_RESTART
        lrs.append(MIN_LR + 0.5 * (BASE_LR - MIN_LR) * (1 + math.cos(math.pi * t / T_RESTART)))

plt.figure(figsize=(10, 3))
plt.plot(lrs, color='steelblue')
plt.axvspan(0, WARMUP_EPOCHS, alpha=0.15, color='orange', label='Warmup')
plt.title('LR Schedule: Linear Warmup + Cosine Annealing with Warm Restarts')
plt.xlabel('Epoch'); plt.ylabel('Learning Rate'); plt.legend()
plt.tight_layout(); plt.show()


## 6. Training: CutMix/Mixup + Label Smoothing

**Label Smoothing** softens one-hot targets:
$$\tilde{y}_k = y_k(1-\varepsilon) + \varepsilon/K, \quad \varepsilon=0.1,\; K=100$$

This prevents over-confident predictions, improves calibration, and acts as a regulariser.
CutMix and Mixup are applied randomly each batch (50/50 probability).


In [ ]:
model_opt.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=BASE_LR),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# ── tf.data pipeline ─────────────────────────────────────────────────────
def augment_fn(image, label):
    image = data_augmentation(tf.expand_dims(image, 0))[0]
    return image, label

def build_dataset(x, y, training=True, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(len(x), reshuffle_each_iteration=True)
        ds = ds.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = build_dataset(x_train, y_train, training=True)
val_ds   = build_dataset(x_test,  y_test,  training=False)

# ── Custom train step with CutMix/Mixup ──────────────────────────────────
@tf.function
def train_step(images, labels):
    use_cutmix = tf.random.uniform(()) > 0.5
    if use_cutmix:
        mixed_imgs, label_info = cutmix(images, labels)
    else:
        mixed_imgs, label_info = mixup(images, labels)
    with tf.GradientTape() as tape:
        preds = model_opt(mixed_imgs, training=True)
        loss  = mixed_loss(label_info, preds, NUM_CLASSES, SMOOTHING)
    grads = tape.gradient(loss, model_opt.trainable_variables)
    model_opt.optimizer.apply_gradients(zip(grads, model_opt.trainable_variables))
    return loss

# ── Training loop ────────────────────────────────────────────────────────
history_opt = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': [], 'lr': []}
best_val_acc = 0.0
patience_ctr = 0
PATIENCE = 25

print('Starting optimised ResNet training...')
for epoch in range(EPOCHS):
    # Update LR
    if epoch < WARMUP_EPOCHS:
        new_lr = BASE_LR * (epoch + 1) / WARMUP_EPOCHS
    else:
        t = (epoch - WARMUP_EPOCHS) % T_RESTART
        new_lr = MIN_LR + 0.5 * (BASE_LR - MIN_LR) * (1 + math.cos(math.pi * t / T_RESTART))
    model_opt.optimizer.learning_rate.assign(new_lr)

    # Train one epoch
    batch_losses = [float(train_step(imgs, lbls)) for imgs, lbls in train_ds]

    # Evaluate (use no-aug dataset for clean metric)
    clean_train_ds = build_dataset(x_train, y_train, training=False, batch_size=512)
    train_metrics = model_opt.evaluate(clean_train_ds, verbose=0)
    val_metrics   = model_opt.evaluate(val_ds, verbose=0)

    history_opt['loss'].append(np.mean(batch_losses))
    history_opt['accuracy'].append(train_metrics[1])
    history_opt['val_accuracy'].append(val_metrics[1])
    history_opt['val_loss'].append(val_metrics[0])
    history_opt['lr'].append(new_lr)

    if val_metrics[1] > best_val_acc:
        best_val_acc = val_metrics[1]
        model_opt.save('best_resnet_opt.keras')
        patience_ctr = 0
    else:
        patience_ctr += 1

    print(f'Epoch {epoch+1:03d}/{EPOCHS} | '
          f'loss={np.mean(batch_losses):.4f} | '
          f'acc={train_metrics[1]:.4f} | '
          f'val_acc={val_metrics[1]:.4f} | '
          f'lr={new_lr:.2e}')

    if patience_ctr >= PATIENCE:
        print(f'Early stopping at epoch {epoch + 1}')
        break

model_opt = tf.keras.models.load_model('best_resnet_opt.keras')
_, accuracy_opt = model_opt.evaluate(val_ds, verbose=0)
print(f'\n==> Best Validation Accuracy (Optimised ResNet): {accuracy_opt:.4f}')


## 7. Transfer Learning: Fine-Tuned EfficientNetV2-S (Two-Phase)

**Improvements over A4 (frozen VGG16):**

| Feature | A4 baseline | Optimised |
|---|---|---|
| Backbone | VGG16 (138M params) | EfficientNetV2-S (21M params) |
| Backbone training | Fully frozen | Phase 2: top 50 layers unfrozen |
| Input size | 96×96 | 128×128 |
| Head | Dense(256) → Dense(100) | BN → Dense(512) → Dense(100) |
| Augmentation | Basic flips/shifts | Full pipeline |

**Two-phase strategy prevents catastrophic forgetting:**
- Phase 1 (frozen backbone): head learns to map pretrained features to CIFAR-100 classes
- Phase 2 (top-50 layers unfrozen, LR=1e-5): fine-tunes high-level features for CIFAR-100


In [ ]:
IMG_SIZE = 128

def resize_ds(x, y, size=IMG_SIZE, training=True, batch_size=64):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(len(x), reshuffle_each_iteration=True)
    def _proc(img, lbl):
        img = tf.image.resize(img, (size, size))
        if training:
            img = data_augmentation(tf.expand_dims(img, 0))[0]
        return img, lbl
    return ds.map(_proc, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

eff_train_ds = resize_ds(x_train, y_train, training=True)
eff_val_ds   = resize_ds(x_test,  y_test,  training=False)

# ── Build EfficientNetV2-S model ──────────────────────────────────────────
backbone = EfficientNetV2S(include_top=False, weights='imagenet',
                           input_shape=(IMG_SIZE, IMG_SIZE, 3),
                           include_preprocessing=False)
backbone.trainable = False

inp = layers.Input(shape=(32, 32, 3))
x   = layers.UpSampling2D((IMG_SIZE // 32, IMG_SIZE // 32), interpolation='bilinear')(inp)
x   = backbone(x, training=False)
x   = layers.GlobalAveragePooling2D()(x)
x   = layers.BatchNormalization()(x)
x   = layers.Dense(512, activation='relu',
                   kernel_regularizer=regularizers.l2(1e-4))(x)
x   = layers.Dropout(0.4)(x)
out = layers.Dense(100, activation='softmax')(x)

model_effnet = models.Model(inp, out, name='EfficientNetV2S_CIFAR100')

# ── Phase 1: Train head only (backbone frozen) ────────────────────────────
model_effnet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)
print('=== Phase 1: Head training (backbone frozen) ===')
hist_p1 = model_effnet.fit(
    eff_train_ds, epochs=20, validation_data=eff_val_ds,
    callbacks=[callbacks.EarlyStopping('val_accuracy', patience=8,
                                        restore_best_weights=True, verbose=1)],
    verbose=1
)
_, acc_p1 = model_effnet.evaluate(eff_val_ds, verbose=0)
print(f'Phase 1 best val_accuracy: {acc_p1:.4f}')

# ── Phase 2: Unfreeze top 50 backbone layers ──────────────────────────────
backbone.trainable = True
for layer in backbone.layers[:-50]:
    layer.trainable = False

model_effnet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),   # very small LR for fine-tuning
    loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)
print('\n=== Phase 2: Fine-tuning top 50 backbone layers (LR=1e-5) ===')
hist_p2 = model_effnet.fit(
    eff_train_ds, epochs=30, validation_data=eff_val_ds,
    callbacks=[
        callbacks.EarlyStopping('val_accuracy', patience=10,
                                restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau('val_loss', factor=0.5,
                                    patience=5, min_lr=1e-7, verbose=1)
    ],
    verbose=1
)
_, accuracy_effnet = model_effnet.evaluate(eff_val_ds, verbose=0)
print(f'\n==> Final Accuracy (Fine-tuned EfficientNetV2-S): {accuracy_effnet:.4f}')


## 8. Noise Robustness Evaluation

Same Gaussian noise protocol as A2: $\sigma^2 = 0.05$, noise added in normalised space.


In [ ]:
def add_gaussian_noise(images, variance=0.05):
    noise = np.random.normal(0.0, np.sqrt(variance), images.shape).astype('float32')
    # Clip within observed min/max (not [0,1] since data is now normalised)
    return np.clip(images + noise, images.min(), images.max())

x_test_noisy = add_gaussian_noise(x_test, variance=0.05)

val_noisy_ds  = build_dataset(x_test_noisy, y_test, training=False)
eff_noisy_ds  = resize_ds(x_test_noisy, y_test, training=False)

_, acc_opt_noisy  = model_opt.evaluate(val_noisy_ds, verbose=0)
_, acc_eff_noisy  = model_effnet.evaluate(eff_noisy_ds, verbose=0)

print('\n' + '='*68)
print(f'{"Model":<38} {"Clean":>7} {"Noisy":>7} {"Drop":>8}')
print('='*68)
print(f'{"Baseline Scratch CNN (A1)":<38} {0.3825:>7.4f} {0.0611:>7.4f} {-84.07:>7.1f}%')
print(f'{"Baseline VGG16 Frozen (A4)":<38} {0.4217:>7.4f} {0.0266:>7.4f} {-93.69:>7.1f}%')
print(f'{"Optimised ResNet+SE+CutMix":<38} {accuracy_opt:>7.4f} {acc_opt_noisy:>7.4f} '
      f'{(acc_opt_noisy - accuracy_opt) / accuracy_opt * 100:>7.1f}%')
print(f'{"Fine-tuned EfficientNetV2-S":<38} {accuracy_effnet:>7.4f} {acc_eff_noisy:>7.4f} '
      f'{(acc_eff_noisy - accuracy_effnet) / accuracy_effnet * 100:>7.1f}%')
print('='*68)


## 9. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].plot(history_opt['accuracy'], label='Train')
axes[0, 0].plot(history_opt['val_accuracy'], label='Val')
axes[0, 0].set_title('Optimised ResNet+SE — Accuracy')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(history_opt['loss'], label='CutMix/Mixup train loss')
axes[0, 1].plot(history_opt['val_loss'], label='Val loss')
axes[0, 1].set_title('Optimised ResNet+SE — Loss')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

p1_acc = hist_p1.history['val_accuracy']
p2_acc = hist_p2.history['val_accuracy']
full_train = hist_p1.history['accuracy'] + hist_p2.history['accuracy']
full_val   = p1_acc + p2_acc
axes[1, 0].plot(full_train, label='Train')
axes[1, 0].plot(full_val, label='Val')
axes[1, 0].axvline(len(p1_acc) - 1, color='red', linestyle='--', label='Phase 2 start')
axes[1, 0].set_title('EfficientNetV2-S — Accuracy')
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

p1_loss = hist_p1.history['val_loss']
p2_loss = hist_p2.history['val_loss']
axes[1, 1].plot(hist_p1.history['loss'] + hist_p2.history['loss'], label='Train')
axes[1, 1].plot(p1_loss + p2_loss, label='Val')
axes[1, 1].axvline(len(p1_loss) - 1, color='red', linestyle='--', label='Phase 2 start')
axes[1, 1].set_title('EfficientNetV2-S — Loss')
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

for ax in axes.flat:
    ax.set_xlabel('Epoch')
plt.suptitle('Optimised Model Training Curves', fontsize=14)
plt.tight_layout(); plt.show()


## 10. Summary of All Optimisations

| Optimisation | Technique | Why It Helps |
|---|---|---|
| **Normalisation** | Per-channel mean/std | Centred activations, faster convergence |
| **Architecture** | ResNet + SE attention | Residual gradients + channel focus |
| **BatchNorm** | After every conv | Reduces internal covariate shift |
| **CutMix + Mixup** | Mixed samples/labels | Smoother decision boundary, less overfitting |
| **LR Schedule** | Cosine restart + warmup | Escapes local minima, stable early training |
| **Label Smoothing** | ε=0.1 | Prevents over-confident predictions |
| **EfficientNetV2-S fine-tuning** | Two-phase training | Best of pretrained features + domain adaptation |

### Expected Accuracy Targets

| Model | Expected Accuracy | vs. Baseline (42%) |
|---|---|---|
| Optimised ResNet+SE | ~60–65% | +18–23 pp |
| Fine-tuned EfficientNetV2-S | ~70–75% | +28–33 pp |
